# 🐍 Clase 8 · El framework: Strategy + Backtest

> El corazón del curso: una estrategia solo reacciona al libro y devuelve acciones. El Backtest la cablea con el mercado y el portfolio. Cualquier estrategia se enchufa igual.

**Hoy construyes:** interfaz Strategy (ABC) y el runner Backtest.

### Cómo funciona este cuaderno

1. Escribe tu respuesta en la celda de código.
2. Debajo hay una **✅ comprobación plegada**: ejecútala con `Shift+Enter` para validarte (despliégala si quieres ver el `assert`).
3. ¿Atascado? Abre **💡 Ver solución**.

**Núcleo:** los primeros (en clase) · **Si vamos bien:** el resto · **Más:** el cuaderno de auxiliares.

### 1. Tu primera estrategia

Define `BuyOnce(Strategy)`: en el primer libro envía una market buy de 0.5; después nada.

<sub>practicas: heredar de Strategy</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType
class BuyOnce(Strategy):
    def __init__(self):
        self.done = False
    def on_book_update(self, book):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
from exchange import Market, Backtest
r = Backtest(Market.sample(), BuyOnce()).run()
assert r.n_fills >= 1 and r.final_position > 0
print('ok  fills=%d pos=%.2f' % (r.n_fills, r.final_position))

<details>
<summary>💡 Ver solución</summary>

```python
class BuyOnce(Strategy):
    def __init__(self):
        self.done = False
    def on_book_update(self, book):
        if self.done:
            return []
        self.done = True
        return [NewOrder(Order('BTCUSDT', Side.BUY, 0.5, order_type=OrderType.MARKET))]
```

</details>

### 2. Corre el Backtest

Corre `BuyOnce` con `Backtest(Market.sample(), BuyOnce()).run()`. Guarda `result` y `equity`.

<sub>practicas: Backtest.run</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class BuyOnce(Strategy):
    def __init__(self): self.done=False
    def on_book_update(self, book):
        if self.done: return []
        self.done=True
        return [NewOrder(Order('BTCUSDT', Side.BUY, 0.5, order_type=OrderType.MARKET))]
result = None
equity = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert result.n_steps == 500
assert isinstance(equity, float)
print('ok ', result)

<details>
<summary>💡 Ver solución</summary>

```python
result = Backtest(Market.sample(), BuyOnce()).run()
equity = result.final_equity
```

</details>

### 3. Reacciona a tus fills

Define `CountingBuyer(Strategy)` que cuente sus fills en `self.n` vía `on_fill`. Compra 0.1 cada paso.

<sub>practicas: el hook on_fill</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class CountingBuyer(Strategy):
    def __init__(self):
        self.n = 0
    def on_book_update(self, book):
        return [NewOrder(Order('BTCUSDT', Side.BUY, 0.1, order_type=OrderType.MARKET))]
    def on_fill(self, fill):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
s = CountingBuyer()
Backtest(Market.sample(), s).run()
assert s.n >= 1, 'on_fill debe haberse llamado'
print('ok  on_fill llamado %d veces' % s.n)

<details>
<summary>💡 Ver solución</summary>

```python
class CountingBuyer(Strategy):
    def __init__(self):
        self.n = 0
    def on_book_update(self, book):
        return [NewOrder(Order('BTCUSDT', Side.BUY, 0.1, order_type=OrderType.MARKET))]
    def on_fill(self, fill):
        self.n += 1
```

</details>

### 4. Polimorfismo: cambia la estrategia, no el runner

Define `SellOnce` (igual que BuyOnce pero vende). Córrela en el MISMO Backtest. Guarda `pos` (debe ser < 0).

<sub>practicas: intercambiar subclases</sub>

In [ ]:
from exchange import Strategy, NewOrder, Order, Side, OrderType, Market, Backtest
class SellOnce(Strategy):
    def __init__(self): self.done=False
    def on_book_update(self, book):
        pass

pos = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert pos < 0, 'SellOnce deja posición corta — mismo runner, otra estrategia'
print('ok  pos=%.2f' % pos)

<details>
<summary>💡 Ver solución</summary>

```python
class SellOnce(Strategy):
    def __init__(self): self.done=False
    def on_book_update(self, book):
        if self.done: return []
        self.done=True
        return [NewOrder(Order('BTCUSDT', Side.SELL, 0.5, order_type=OrderType.MARKET))]
pos = Backtest(Market.sample(), SellOnce()).run().final_position
```

</details>

## Cierre

Escribe una subclase de Strategy y enchúfala al mismo Backtest. Eso es polimorfismo, y es lo que hace todo modular.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.